In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
from src.agents.agent_0 import Agent0
agent_0_tools_desc = {'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary"',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "deepseek-r1:7b",['kb_agent','adv_agent'])



In [ ]:
user_prompt = "Need an adversary. Assume you are a military strategist playing the role of an adversary in a war game against me. Consider we are on open terrain. My move: I have my cavalry brigade making a pincer move on your forces. What is your move to counter mine?"
response = agent_0.agent_0_chat(user_prompt)

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent

model = "gemma3:4b"
knowledge_bases_desc = {#'physics_kb':'a knowledge base with information related to physics',
              #'mathematics_kb':'a knowledge base with information related to mathematics',
              #'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }



kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

In [ ]:


user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2

import numpy as np
from src.utils.llmp_utils import llmp_call

def judge(moves):
    
    play = ''
    for move in moves:
        #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
        play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
        
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = play + '\n\n Evaluate the game. Determine the status and advantage of each player. You are a JUDGE, you are not part of the game.'
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

def random_event(dialogue):
    
    interactions = "\n".join(dialogue)
    random_events_system_prompt = 'You are a random events generator. Your tasks is to choose a random event that can happen that will affect the decisions. You are provided with a sequence of plays, you need to select a random event that can affect those plays. You are direct you only provide the needed text, no formalities, no greetings, nothing.'    
    random_event_prompt = interactions + '\n\n Considering this game, provide a random event that can affect the game and force the players to adapt. You must inform what is the effect of the random event on the players. Provide me only the event and effect on players. No unnecessary text! Provide the answer in markdown of the style **<event>**. \n**EFFECT ON PLAYER 1**: \n<effect_player_1>. **EFFECT ON PLAYER 2**: <effect_player_2>'
    judge_response = llmp_call(random_event_prompt, random_events_system_prompt, model,temperature=0.5, src='random_event_generator')
    return judge_response['message']['content']

def sim_agent(user_prompt,iterations):
    
    moves = {}
    dialogue = []

    moves['opening_move'] = user_prompt

    for i in range(iterations):
        print(f"\nTurn {i}")
        

        if i == 0:
            # Start the dialogue with opening
            dialogue.append(f"Opening: {moves['opening_move']}")
            
            # Simulate generating move_adv_1_0 based on just the opening
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print("Prompt to generate move_adv_1_0:\n", prompt)

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            

        else:
            if np.random.random() < 1:
                _random_event = random_event(dialogue)
                dialogue.append(f"\n**Random event**: {_random_event} \n")
            # Use the full dialogue to generate your next move
            prompt = "\n".join(dialogue) + "\n You are Player 2. How will you counter it Player 1 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_2_{i-1}:\n{prompt}")

            # CADV response
            moves[f'move_adv_2_{i-1}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 2 did: {moves[f'move_adv_2_{i-1}']}")

            # Now generate adversary move based on updated dialogue
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_1_{i}:\n{prompt}")

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            
        judge_eval = judge(moves)
        
    return moves,dialogue,judge_eval


In [ ]:
moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridge or rise – offering better observation and defensive potential.

2. **Establish a Defensive Perimeter (Phase 2 - 2-3 Turns):** As t

In [ ]:
moves

{'opening_move': 'Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'move_adv_1_0': 'Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my

In [ ]:
dialogue

['Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my scout pl

In [ ]:
print("\n".join(dialogue))

Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?
Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridg

In [ ]:
print(judge_eval)

Okay, let’s assess the situation after this extended exchange. This has been a remarkably dynamic and well-executed game of strategic maneuvering. Here’s my evaluation:

**Overall Status:** The game is in a state of heightened instability. The introduction of the flash flood has dramatically shifted the landscape, forcing both players to adapt their strategies on the fly. Neither player has gained a decisive advantage, but the situation is now far more complex and unpredictable.

**Player 1 (Advantage: Slight)**

* **Strengths:** Player 1 has demonstrated a strong ability to react to unexpected events. The rapid damage assessment, floodwater diversion, and logistical reinforcement are all hallmarks of a well-organized and adaptable command. The continuous CAS requests suggest a proactive approach to exploiting vulnerabilities.
* **Weaknesses:** Player 1’s initial offensive push was disrupted, and they’re now primarily focused on damage control and logistical support. They haven’t yet m

In [ ]:
sim_number = 3

moves_comb = []
dialogue_comb = []
judge_eval_comb = []
for sim in range(sim_number):
    moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)
    moves_comb.append(moves)
    dialogue_comb.append(dialogue)
    judge_eval_comb.append(judge_eval)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, but it’s also predictable. Here’s my immediate counter-move, broken down into steps:

**Phase 1: Immediate Reaction (Turn 1)**

1.  **Disrupt the Pincer:** I’m not going to let them fully execute the pincer. My initial move is to deploy a dispersed, mobile force – a mixed unit of light armored vehicles (LAVs) and rapid reaction forces (RRFs) – to target the flanks of the mechanized brigade. Specifically, I’ll focus fire on the weaker, exposed elements of the flanking units. The goal is to inflict immediate casualties and disrupt their formation.
2.  **Smoke Screen:** Simultaneously, I’ll deploy a limited smoke screen – likely utilizing drones or hand

In [ ]:
for eval in judge_eval_comb:
    print(f"\n **CHANGE SIM**\n{eval}")


 **CHANGE SIM**
Okay, let’s analyze the situation as of Turn 4.

**Overall Assessment:**

The game has devolved into a classic attritional conflict, heavily influenced by the unpredictable element of the sandstorm. Both Player 1 and Player 2 are demonstrating tactical awareness and adaptability, but Player 2 currently holds a slight advantage due to their skillful exploitation of the storm’s chaos.

**Player 1’s Status:**

*   **Strengths:** Player 1 is exhibiting a solid defensive strategy, prioritizing perimeter defense, smoke screen deployment, and targeted drone interdiction. Their focus on suppressing enemy movements with indirect fire is a reasonable response to Player 2’s aggressive pushes. The emphasis on information warfare (drone interdiction) is also a smart move.
*   **Weaknesses:** Player 1’s reliance on indirect fire makes them vulnerable to counter-fire. Their defensive perimeter, while well-organized, is relatively static and doesn’t offer significant offensive capabil

In [ ]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent

In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2
moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt, iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter Move – Immediate Steps:**

1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.

2. **Rapid Scout Deployment:** Simultaneously, I order my scout platoon to rapidly deploy to the *flanking* side of the pincer. This means they’ll move to exploit the gaps in Player 2’s formation. The goal is t

In [ ]:
print(judge_eval)

**Judgment:**

**Current Status:** The game has entered a highly dynamic and disadvantageous phase for both players due to the persistent and severe sandstorm. Visibility is severely limited, significantly impacting reconnaissance, movement, and targeting capabilities. The reduced movement speed of mechanized units further compounds the problem.

**Advantage Assessment:**

*   **Player 2 (Adv_2) – Slight Advantage:** Despite the storm’s impact on both sides, Player 2 currently holds a *slight* advantage. This is primarily due to their immediate and effective response to the storm. They prioritized establishing a defensive strongpoint and aggressively utilizing thermal imaging to pinpoint Player 1’s movements. Their proactive approach, coupled with the storm’s impact on Player 1’s ability to effectively scout and target, has allowed them to maintain a degree of situational awareness and control.

*   **Player 1 (Adv_1) – Slight Disadvantage:** Player 1’s response, while demonstrating a 

## Agent 0 integration

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.agent_0 import Agent0

agent_0_tools_desc = {
    'Simulation Agent':'a simulation agent that simulates a game between two players. Triggered by command "Simulate a scenario.". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!'
              }


agent_0 = Agent0(agent_0_tools_desc, "gemma3:4b",['kb_agent','adv_agent','sim_agent'])

Initializing Agents!
Agents are ready for your use!


In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
from src.utils.llmp_utils import llmp_call
model = "gemma3:4b"

knowledge_bases_desc = {
    #'physics_kb':'a knowledge base with information related to physics',
    #'mathematics_kb':'a knowledge base with information related to mathematics',
    #'economics_kb':'a knowledge base with information related to economics and business',,
    'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product.'
user_prompt = 'I am attacking your position using a pincer movement with my tank division. We are in 21st century'
iterations = 2
#comb_dialogue,diaglogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations)
final_output,dialogue,judge_eval,moves = sim_agent.sim_agent(user_prompt,iterations,1,structured = True)

{'type': 'object', 'properties': {'Player': {'type': 'string'}, 'moves': {'type': 'object', 'additionalProperties': {'type': 'string'}}}, 'required': ['Player', 'moves']}

Turn 0
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
{'type': 'object', 'properties': {'Random Event': {'type': 'string'}, 'Effect on Player 1': {'type': 'string'}, 'Effect on Player 2': {'type': 'string'}}, 'required': ['Random Event', 'Effect on Player 1', 'Effect on Player 2']}
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Pro

In [5]:
print(final_output)

 ### Opening move:  
 I am attacking your position using a pincer movement with my tank division. We are in 21st century
 ---
 ### Player 1 did:
 {
"Player": "Player 1",
"moves": {
    "1": "Implement Defensive Posturing: Immediately order a defensive perimeter around your key objectives – the bridge and any nearby strongpoints. Prioritize establishing a layered defense, utilizing terrain to your advantage.",
    "2": "Counter-Attack Probe: Send a small reconnaissance force – likely a scout car or a squad of infantry – to probe the flanks of Player 2’s pincer. The goal is to identify the exact composition and strength of their attacking force.",
    "3": "Reinforce Vulnerable Sectors: Based on the reconnaissance findings, rapidly reinforce the sectors most threatened by the pincer movement. This might involve deploying additional infantry, artillery support, or even calling in air support if available.",
    "4": "Exploit Terrain: Utilize any natural obstacles – woods, rivers, hills – 

In [6]:
dialogue

['Player 2 did: I am attacking your position using a pincer movement with my tank division. We are in 21st century',
 'Player 1 did: {\n"Player": "Player 1",\n"moves": {\n    "1": "Implement Defensive Posturing: Immediately order a defensive perimeter around your key objectives – the bridge and any nearby strongpoints. Prioritize establishing a layered defense, utilizing terrain to your advantage.",\n    "2": "Counter-Attack Probe: Send a small reconnaissance force – likely a scout car or a squad of infantry – to probe the flanks of Player 2’s pincer. The goal is to identify the exact composition and strength of their attacking force.",\n    "3": "Reinforce Vulnerable Sectors: Based on the reconnaissance findings, rapidly reinforce the sectors most threatened by the pincer movement. This might involve deploying additional infantry, artillery support, or even calling in air support if available.",\n    "4": "Exploit Terrain: Utilize any natural obstacles – woods, rivers, hills – to disr

In [7]:
import json
formated_dialogue = [dialogue[0].replace('Player 2 did:','Opening Move:')]
for response in dialogue[1:]:
    response = response.strip('Player 2 did:')
    response = response.strip('Player 1 did:')
    response = response.strip('\n**Random event**:')
    data = json.loads(response)
    formated_dialogue.append(data)
formated_judge_eval = json.loads(judge_eval)

In [2]:
# GRAND SIM


num_sims = 2
iterations = 3
rand_event_chance = 0.7
user_prompt = 'I am attacking your position using a pincer movement with my tank division. We are in 21st century'


final_outputs = []
dialogues = []
judge_evals = []
moves_list = []
for sim in range(num_sims):
    print(f"\n\n\n\n\n\n\n**SIMULATION {sim}**\n\n\n")
    final_output,dialogue,judge_eval,moves = sim_agent.sim_agent(user_prompt,iterations,rand_event_chance,structured = True)
    final_outputs.append(final_output)
    dialogues.append(dialogue)
    judge_evals.append(judge_eval)
    moves_list.append(moves)

    








**SIMULATION 0**



{'type': 'object', 'properties': {'Player': {'type': 'string'}, 'moves': {'type': 'object', 'additionalProperties': {'type': 'string'}}}, 'required': ['Player', 'moves']}

Turn 0
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
{'type': 'object', 'properties': {'Random Event': {'type': 'string'}, 'Effect on Player 1': {'type': 'string'}, 'Effect on Player 2': {'type': 'string'}}, 'required': ['Random Event', 'Effect on Player 1', 'Effect on Player 2']}
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model..

In [3]:
print(judge_evals[0])

{
  "Game Summary": "The conflict is characterized by a rapid, iterative cycle of offensive and defensive maneuvers, heavily influenced by the unexpected meteor shower event. Both sides are aggressively pursuing tactical gains, utilizing combined arms tactics and electronic warfare. Player 1 initially gained an advantage through rapid reconnaissance and coordinated assaults, but Player 2’s defensive shifts and counter-attacks have significantly slowed Player 1’s momentum. The meteor shower has introduced a significant element of chaos and disruption, impacting both sides’ capabilities.",
  "Player 1 Status": {
    "Actions": [
      "Aggressive Reconnaissance",
      "Counter-Artillery Barrage",
      "Mobile Reserve Assault – Combined Arms",
      "Electronic Warfare – Targeted Disruption",
      "Defensive Consolidation – Layered Defense"
    ],
    "Weaknesses": [
      "Reliance on rapid reconnaissance and initial momentum",
      "Vulnerability to electronic warfare disruption",
 

In [30]:
dialogues[0]

['Player 2 did: I am attacking your position using a pincer movement with my tank division. We are in 21st century',
 'Player 1 did: {\n  "Player": "Player 1",\n  "moves": {\n    "1": "Counter-Attack: Immediate, concentrated assault on the flank where Player 2’s tank division is positioned. Objective: Disrupt the pincer movement and inflict maximum casualties.",\n    "2": "Reinforce Defense: Rapid deployment of infantry and artillery to bolster the threatened sector. Priority: Establish a strong defensive line to resist the pincer’s advance.",\n    "3": "Artillery Barrage: Coordinate a heavy artillery bombardment on Player 2’s tank division’s assembly area. Goal: Suppress their offensive capabilities and inflict heavy damage.",\n    "4": "Mobile Reserve Deployment: Shift a mobile unit – likely infantry supported by armored vehicles – to provide immediate support to the threatened flank. Purpose: Rapid response to any breakthroughs by Player 2’s forces."\n  }\n}\n',
 '\n**Random event**

In [31]:
import json


def process_sim(sim_dialogue,sim_judge_eval):
    
    formated_dialogue = [sim_dialogue[0].replace('Player 2 did:','Opening Move:')]
    for response in sim_dialogue[1:]:
        response = response.strip('Player 2 did:')
        response = response.strip('Player 1 did:')
        response = response.strip('\n**Random event**:')
        data = json.loads(response)
        formated_dialogue.append(data)
    formated_judge_eval = json.loads(sim_judge_eval)
    return formated_dialogue, formated_judge_eval
#formated_dialogue, formated_judge_eval = process_sim(dialogues[2],judge_evals[2])


def post_process_sim(sim_dialogues,sim_judge_evals):
    """Post process the simulation's data.
    Format the dialogue and judge evaluations.

    Args:
        sim_dialogues (_type_): list of dialogues from all simulations (branch)
        sim_judge_evals (_type_): list of judge evals from each simulation (branch judge)
    """
    
    formated_dialogues = []
    formated_judge_evals = []
    failed_sims = []
    for i in range(len(sim_dialogues)):
        try:
            formated_dialogue, formated_judge_eval = process_sim(sim_dialogues[i],sim_judge_evals[i])
            formated_dialogues.append(formated_dialogue)
            formated_judge_evals.append(formated_judge_eval)
        except:
            failed_sims.append(i)
    return formated_dialogues, formated_judge_evals, failed_sims
    

def transform_moves(moves_dict):
    """Transforms moves dictionary to a more structured format.

    Args:
        moves_dict (list): list of moves dictionaries

    Returns:
        _type_: formated moves dictionary
    """
    new_moves = {}
    for _, value in moves_dict["moves"].items():
        if isinstance(value, str) and ":" in value:
            move_name, description = map(str.strip, value.split(":", 1))
            new_moves[move_name] = description
        else:
            new_moves[value] = ""  # fallback if not formatted as expected

    return {
        "Player": moves_dict["Player"],
        "moves": new_moves
    }
    

def transform_moves_list(moves_list):
    """Transforms a list of moves dictionaries to a more structured format.

    Args:
        moves_list (list): list of moves dictionaries

    Returns:
        _type_: formated list of moves dictionaries
    """
    new_moves_list = []
    for moves_dict in moves_list:
        if "Opening Move" in moves_dict:
            new_moves_list.append(moves_dict)
        elif "Random Event" in moves_dict.keys():
            new_moves_list.append(moves_dict)
        else:
            transformed_move = transform_moves(moves_dict)
            new_moves_list.append(transformed_move)
    return new_moves_list

def transform_sims_moves(sims_moves):
    """transform all sims moves to a more structured format

    Args:
        sims_moves (list): list of all simulations moves

    Returns:
        _type_: structured list of all simulations moves
    """
    new_sims_moves = []
    for i in range(len(sims_moves)):
        new_sim_moves = transform_moves_list(sims_moves[i])
        new_sims_moves.append(new_sim_moves)
        
    return new_sims_moves


In [38]:
formated_dialogues, formated_judge_evals, failed_sims = post_process_sim(dialogues,judge_evals)

In [39]:

new_formated_dialogues = transform_sims_moves(formated_dialogues)

In [40]:
formated_dialogues

[['Opening Move: I am attacking your position using a pincer movement with my tank division. We are in 21st century',
  {'Player': 'Player 1',
   'moves': {'1': 'Counter-Attack: Immediate, concentrated assault on the flank where Player 2’s tank division is positioned. Objective: Disrupt the pincer movement and inflict maximum casualties.',
    '2': 'Reinforce Defense: Rapid deployment of infantry and artillery to bolster the threatened sector. Priority: Establish a strong defensive line to resist the pincer’s advance.',
    '3': 'Artillery Barrage: Coordinate a heavy artillery bombardment on Player 2’s tank division’s assembly area. Goal: Suppress their offensive capabilities and inflict heavy damage.',
    '4': 'Mobile Reserve Deployment: Shift a mobile unit – likely infantry supported by armored vehicles – to provide immediate support to the threatened flank. Purpose: Rapid response to any breakthroughs by Player 2’s forces.'}},
  {'Random Event': 'Sudden Dust Storm, visibility reduc

In [7]:
formated_judge_evals

[{'Game Summary': 'The conflict has devolved into a series of aggressive counter-attacks and defensive consolidation. Both sides are attempting to exploit weaknesses and maintain offensive momentum. The initial pincer movement by Player 2 was disrupted by Player 1’s immediate counter-attack and the subsequent dust storm significantly hampered Player 1’s artillery efforts. Player 2’s rapid withdrawal and consolidation followed, but Player 1’s continued reconnaissance and targeted strikes have prevented them from establishing a secure defensive position. The battlefield is characterized by a dynamic, fluid situation with both sides attempting to dictate the terms of engagement.',
  'Player 1 Status': {'Actions': ['Counter-Attack (Initial)',
    'Reinforce Defense',
    'Artillery Barrage',
    'Mobile Reserve Deployment',
    'Aggressive Pursuit',
    'Targeted Artillery Strikes',
    'Combined Arms Assault',
    'Mobile Defense Deployment',
    'Aggressive Reconnaissance'],
   'Weakness

In [139]:
new_formated_dialogues[1][0]

'Opening Move: I am attacking your position using a pincer movement with my tank division. We are in 21st century'

## Experiment converting to DF

In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
from src.utils.llmp_utils import llmp_call
model = "gemma3:4b"

knowledge_bases_desc = {
    #'physics_kb':'a knowledge base with information related to physics',
    #'mathematics_kb':'a knowledge base with information related to mathematics',
    #'economics_kb':'a knowledge base with information related to economics and business',,
    'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [5]:
prompt = 'tell me a story'
system_prompt = ''
src = 'test'
max_gen_lenght = 10
llmp_call(prompt, system_prompt, model,temperature=0.5,src=src,format=None)

{'model': 'gemma3:4b',
 'created_at': '2025-05-14T07:34:38.9556952Z',
 'message': {'role': 'assistant',
  'content': "Okay, here’s a story for you. It’s called “The Cartographer’s Daughter” and I hope you enjoy it:\n\nThe rain in Port Blossom was a constant, grey companion. It clung to the slate roofs, dripped from the overflowing gutters, and seeped into the bones of everyone who lived there. Elara, daughter of the renowned cartographer Silas Blackwood, knew it intimately. She spent her days not sketching fantastical islands and shimmering seas like her father, but meticulously charting the tidal patterns of the harbor, the shifting sands of the beach, and the intricate network of cobblestone streets.\n\nSilas Blackwood was a legend. His maps weren’t just representations of land; they were imbued with a feeling, a sense of the place's soul. He’d spent his life traveling, meticulously documenting the coastlines of the known world, and his maps were prized possessions, sought after by s

In [2]:
# GRAND SIM


num_sims = 5
iterations = 5
rand_event_chance = 0.7
user_prompt = 'I am attacking your position using a pincer movement with my tank division. We are in 21st century'


final_outputs = []
dialogues = []
judge_evals = []
moves_list = []
for sim in range(num_sims):
    print(f"\n\n\n\n\n\n\n**SIMULATION {sim}**\n\n\n")
    final_output,dialogue,judge_eval,moves = sim_agent.sim_agent(user_prompt,iterations,rand_event_chance,structured = True)
    final_outputs.append(final_output)
    dialogues.append(dialogue)
    judge_evals.append(judge_eval)
    moves_list.append(moves)

    








**SIMULATION 0**



{'type': 'object', 'properties': {'Player': {'type': 'string'}, 'moves': {'type': 'object', 'additionalProperties': {'type': 'string'}}}, 'required': ['Player', 'moves']}

Turn 0
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
{'type': 'object', 'properties': {'Random Event': {'type': 'string'}, 'Effect on Player 1': {'type': 'string'}, 'Effect on Player 2': {'type': 'string'}}, 'required': ['Random Event', 'Effect on Player 1', 'Effect on Player 2']}
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model..

## Combined post processing

In [130]:
import json
from itertools import zip_longest
import pandas as pd

def preprocess_sim(sim_dialogue,sim_judge_eval):
    
    formated_dialogue = [sim_dialogue[0].replace('Player 2 did:','Opening Move:')]
    for response in sim_dialogue[1:]:
        response = response.strip('Player 2 did:')
        response = response.strip('Player 1 did:')
        response = response.strip('\n**Random event**:')
        data = json.loads(response)
        formated_dialogue.append(data)
    formated_judge_eval = json.loads(sim_judge_eval)
    return formated_dialogue, formated_judge_eval
#formated_dialogue, formated_judge_eval = process_sim(dialogues[2],judge_evals[2])


def post_process_sim(sim_dialogues,sim_judge_evals):
    """Post process the simulation's data.
    Format the dialogue and judge evaluations.

    Args:
        sim_dialogues (_type_): list of dialogues from all simulations (branch)
        sim_judge_evals (_type_): list of judge evals from each simulation (branch judge)
    """
    
    formated_dialogues = []
    formated_judge_evals = []
    failed_sims = []
    for i in range(len(sim_dialogues)):
        try:
            formated_dialogue, formated_judge_eval = preprocess_sim(sim_dialogues[i],sim_judge_evals[i])
            formated_dialogues.append(formated_dialogue)
            formated_judge_evals.append(formated_judge_eval)
        except:
            failed_sims.append(i)
    return formated_dialogues, formated_judge_evals, failed_sims
    

def transform_moves(moves_dict):
    """Transforms moves dictionary to a more structured format.

    Args:
        moves_dict (list): list of moves dictionaries

    Returns:
        _type_: formated moves dictionary
    """
    new_moves = {}
    for _, value in moves_dict["moves"].items():
        if isinstance(value, str) and ":" in value:
            move_name, description = map(str.strip, value.split(":", 1))
            new_moves[move_name] = description
        else:
            new_moves[value] = ""  # fallback if not formatted as expected

    return {
        "Player": moves_dict["Player"],
        "moves": new_moves
    }
    

def transform_moves_list(moves_list):
    """Transforms a list of moves dictionaries to a more structured format.

    Args:
        moves_list (list): list of moves dictionaries

    Returns:
        _type_: formated list of moves dictionaries
    """
    new_moves_list = []
    for moves_dict in moves_list:
        if "Opening Move" in moves_dict:
            new_moves_list.append(moves_dict)
        elif "Random Event" in moves_dict.keys():
            new_moves_list.append(moves_dict)
        else:
            transformed_move = transform_moves(moves_dict)
            new_moves_list.append(transformed_move)
    return new_moves_list

def transform_sims_moves(sims_moves):
    """transform all sims moves to a more structured format

    Args:
        sims_moves (list): list of all simulations moves

    Returns:
        _type_: structured list of all simulations moves
    """
    new_sims_moves = []
    for i in range(len(sims_moves)):
        new_sim_moves = transform_moves_list(sims_moves[i])
        new_sims_moves.append(new_sim_moves)
        
    return new_sims_moves

def get_opening_move(sim_dialogue):
    """Extract opening move from the formatted simulation.

    Args:
        formatted_sim (list): list of events
    """
    opening_move = sim_dialogue[0]
    event_list = sim_dialogue[1:]
    
    return opening_move, event_list

def fill_na_rand_events(event_list):
    """Generate empty random events

    Args:
        event_list (list): list of events

    Returns:
        list: list of events with empty random events
    """
    new_event_list = []
        
    for i, play in enumerate(event_list):
        if "Player" in play and play["Player"] == "Player 2":
            if not (new_event_list and "Random Event" in new_event_list[-1]):
                # Insert dummy event before Player 2's move
                new_event_list.append({
                    "Random Event": "N/A",
                    "EFFECT ON PLAYER 1": "N/A",
                    "EFFECT ON PLAYER 2": "N/A"
                })
        new_event_list.append(play)
        
    return new_event_list

def get_random_events(event_list):
    """Extract random events from the event list.

    Args:
        event_list (list): list of events

    Returns:
        list: list of random events
    """
    rand_events = []
    play_list = []
    for event in event_list:
        if "Random Event" in event.keys():
            rand_events.append(event)
        elif "Player" in event.keys():
            play_list.append(event)
            
            
            
    return rand_events,play_list

def apply_turns_re(rand_events):
    """Apply turns to the random events.

    Args:
        rand_events (list): list of random events

    Returns:
        list: list of random events with turns
    """
    turned_ra_events = rand_events.copy()
    for i, ra_event in enumerate(turned_ra_events):
        ra_event['turn'] = i+1
            
    return turned_ra_events

def clean_rand_events_list(rand_events):
    """Remove NA random events

    Args:
        rand_events (list): list of turned random events
    """
    
    for rand_event in rand_events:
        if "N/A" in rand_event.values():
            rand_events.remove(rand_event)
    return rand_events

def apply_turns_plays(move_list):
    """Apply turns to the plays.

    Args:
        play_list (list): list of plays

    Returns:
        list: list of plays with turns
    """
    p1_moves = [move for move in move_list if "moves" in move.keys() and move['Player'] == 'Player 1']
    p2_moves = [move for move in move_list if "moves" in move.keys() and move['Player'] == 'Player 2']
    for p1_move in p1_moves:
        p1_move['turn'] = p1_moves.index(p1_move)
    for p2_move in p2_moves:
        p2_move['turn'] = p2_moves.index(p2_move) + 1
    turned_moves = [move for pair in zip_longest(p1_moves, p2_moves) for move in pair if move is not None]
    

    return turned_moves

def get_move_df(turned_move_dict):
    """Get a dataframe from the moves dictionary.

    Args:
        move_dict (dict): moves dictionary

    Returns:
        _type_: dataframe of moves
    """
    df = pd.DataFrame([{
        'Player': turned_move_dict['Player'],
        'move': move,
        'move_desc': desc,
        'turn': turned_move_dict['turn']
    } for move, desc in turned_move_dict['moves'].items()])
    
    return df

def convert_moves_to_df(turned_moves_list):
    """Get a dataframe from the moves list.

    Args:
        moves_list (list): list of moves

    Returns:
        _type_: dataframe of moves
    """
    df = pd.DataFrame()
    for move_dict in turned_moves_list:
        temp_df = get_move_df(move_dict)
        df = pd.concat([df, temp_df], ignore_index=True)
    
    return df

def get_rand_events_df(turned_rand_events_list):
    """Get a dataframe from the random events list.

    Args:
        rand_events_list (list): list of random events

    Returns:
        _type_: dataframe of random events
    """
    df = pd.DataFrame(turned_rand_events_list)
    return df

def combine_dfs(moves_df, rand_event_df,opening_move):
    """Combine the moves dataframe and random events dataframe.

    Args:
        moves_df (_type_): moves dataframe
        rand_event_df (_type_): random events dataframe

    Returns:
        _type_: combined dataframe
    """
    sim_df = pd.DataFrame()
    if rand_event_df is None:
        sim_df = moves_df.copy()
        sim_df['Random Event'] = ''
        sim_df['Effect on Player 1'] = ''
        sim_df['Effect on Player 2'] = ''
    else:
        sim_df = moves_df.merge(rand_event_df, on='turn', how='left')
    sim_df['opening_move'] = opening_move
    
    columns = ['opening_move','turn', 'Player', 'move', 'move_desc', 'Random Event', 'Effect on Player 1', 'Effect on Player 2']
    sim_df = sim_df[columns]
    
    return sim_df
    
def process_sim(formated_sim_dialogue):
    """Simulation processing into dataframe pipeline

    Args:
        formated_sim_dialogue (list): formated simulation dialogue
    """
    opening_move, event_list = get_opening_move(formated_sim_dialogue)
    event_list = fill_na_rand_events(event_list)
    rand_events,move_list = get_random_events(event_list)
    turned_rand_events = apply_turns_re(rand_events)
    turned_rand_events = clean_rand_events_list(turned_rand_events)
    turned_moves = apply_turns_plays(move_list)
    moves_df = convert_moves_to_df(turned_moves)
    rand_events_df = get_rand_events_df(turned_rand_events)
    sim_df = combine_dfs(moves_df, rand_events_df,opening_move)
    
    return sim_df
    
def process_grand_sim(formated_sims_dialogues):
    """Combines all simulations into grand sim

    Args:
        formated_sims_dialogues (list): list of simulation events
    """
    grand_sim_df = pd.DataFrame()
    for sim in formated_sims_dialogues:
        index = formated_sims_dialogues.index(sim)
        sim_df = process_sim(sim)
        sim_df['sim'] = index
        grand_sim_df = pd.concat([grand_sim_df, sim_df], ignore_index=True)
        
    columns = ['opening_move','sim','turn','Player','move','move_desc','Random Event','Effect on Player 1','Effect on Player 2']
    grand_sim_df = grand_sim_df[columns]
        
    return grand_sim_df


def process_judge_df(judge_eval):

    player_summary = {
        "player_1_weaknesses": judge_eval["Player 1 Status"]["Weaknesses"],
        "player_1_strengths": judge_eval["Player 1 Status"]["Strengths"],
        "player_1_overall_status": judge_eval["Player 1 Status"]["Overall Status"],
        "player_2_weaknesses": judge_eval["Player 2 Status"]["Weaknesses"],
        "player_2_strengths": judge_eval["Player 2 Status"]["Strengths"],
        "player_2_overall_status": judge_eval["Player 2 Status"]["Overall Status"],
        "outcome": judge_eval["Outcome"],
        "advantage": judge_eval["Advantage"],
    }

    # Convert to DataFrame
    df_summary = pd.DataFrame([player_summary])
    return df_summary

def process_judge_evals(formated_judge_evals):
    """Process the judge evaluations data.

    Args:
        formated_judge_evals (_type_): _description_

    Returns:
        _type_: _description_
    """

    grand_judge_df = pd.DataFrame()
    for judge in formated_judge_evals:
        index = formated_judge_evals.index(judge)
        judge_df = process_judge_df(judge)
        judge_df['sim'] = index
        grand_judge_df = pd.concat([grand_judge_df, judge_df], ignore_index=True)
    return grand_judge_df

In [131]:
formated_dialogues, formated_judge_evals, failed_sims = post_process_sim(dialogues,judge_evals)
new_formated_dialogues = transform_sims_moves(formated_dialogues)

grand_sim_df = process_grand_sim(new_formated_dialogues)

In [132]:
grand_sim_judges_df = process_judge_evals(formated_judge_evals)

In [134]:
grand_sim_judges_df.to_csv('grand_sim_judges.csv', index=False)